# Entropic plans as the temperature changes

This notebook generates `fig:sinkhorn-plan-epsilon`.  It uses the canonical red/blue point clouds from the matching and Kantorovich figures, and solves
$$
\min_{P\in U(a,b)} \langle C,P\rangle + \varepsilon\,\mathrm{KL}(P\mid a\otimes b)
$$
for four fixed values of `epsilon`.  The point is not to benchmark Sinkhorn, but to visualize how entropy diffuses the coupling.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/mpl-ot4ml")

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np
import ot
from matplotlib.collections import LineCollection
from matplotlib.colors import LinearSegmentedColormap, to_rgb
from matplotlib.patches import Polygon

from figure_style import (
    BLUE,
    DIRAC_MARKER_SIZE,
    GRAY,
    LIGHT_GRAY,
    ORANGE,
    RED,
    VIOLET,
    box_axes,
    canonical_matching_clouds,
    figure_dir,
    interp_color,
    padded_limits,
    remove_axes,
    save_pdf,
    setup_matplotlib,
)

setup_matplotlib()
np.random.seed(0)

POINT_SIZE = 10.5


def active_pairs(P, rel_threshold=0.18, max_pairs=180):
    order = np.argsort(P.ravel())[::-1]
    pairs = []
    cutoff = rel_threshold * P.max()
    for flat in order:
        mass = float(P.ravel()[flat])
        if mass < cutoff:
            break
        i, j = np.unravel_index(flat, P.shape)
        pairs.append((int(i), int(j), mass))
        if len(pairs) >= max_pairs:
            break
    return pairs


def draw_clouds(ax, x, y, *, alpha=1.0, size=POINT_SIZE, zorder=4):
    ax.scatter(x[:, 0], x[:, 1], s=size, marker="o", color=RED, alpha=alpha, edgecolor="none", zorder=zorder)
    ax.scatter(y[:, 0], y[:, 1], s=size, marker="o", color=BLUE, alpha=alpha, edgecolor="none", zorder=zorder)


def draw_segments(ax, x, y, pairs, *, color=VIOLET, min_width=0.10, max_width=1.15, alpha=0.56, zorder=1):
    if not pairs:
        return
    masses = np.array([m for _, _, m in pairs], dtype=float)
    rel = masses / max(masses.max(), 1e-15)
    segments = [[x[i], y[j]] for i, j, _ in pairs]
    base = np.array(to_rgb(color))
    colors = [(*base, min(alpha * (0.18 + 0.82 * np.sqrt(r)), 0.88)) for r in rel]
    widths = min_width + (max_width - min_width) * np.sqrt(rel)
    ax.add_collection(LineCollection(segments, colors=colors, linewidths=widths, zorder=zorder))


def set_cloud_limits(ax, x, y, pad=0.18):
    (xmin, xmax), (ymin, ymax) = padded_limits(np.vstack([x, y]), pad=pad)
    span = max(xmax - xmin, ymax - ymin)
    cx, cy = (xmin + xmax) / 2, (ymin + ymax) / 2
    ax.set_xlim(cx - span / 2, cx + span / 2)
    ax.set_ylim(cy - span / 2, cy + span / 2)
    ax.set_aspect("equal")
    remove_axes(ax)

## Canonical point clouds

All four panels use the same source and target locations.  The displayed line set is thresholded only for readability; the computed entropic plan is strictly positive for every `epsilon > 0`.

In [ ]:
fig_name = "sinkhorn-plan-epsilon"
out = figure_dir(fig_name)

x, y, labels = canonical_matching_clouds(seed=2027, n_source=24)
a = np.ones(len(x)) / len(x)
b = np.ones(len(y)) / len(y)
C = ot.dist(x, y, metric="euclidean") ** 2
C = C / np.median(C[C > 0])
epsilons = [
    (0.008, "eps-0p008.pdf", 0.55, 28),
    (0.025, "eps-0p025.pdf", 0.34, 50),
    (0.08, "eps-0p080.pdf", 0.18, 105),
    (0.25, "eps-0p250.pdf", 0.095, 220),
]

## Exported coupling panels

Segments have width and opacity proportional to the visible transported mass.  Small `epsilon` gives a nearly sparse geometry; large `epsilon` spreads mass across many source-target pairs.

In [ ]:
for epsilon, filename, rel_threshold, max_pairs in epsilons:
    K = (a[:, None] * b[None, :]) * np.exp(-C / epsilon)
    plan = ot.sinkhorn(a, b, K, reg=1.0, method="sinkhorn_log", numItermax=10000, stopThr=1e-12)
    pairs = active_pairs(plan, rel_threshold=rel_threshold, max_pairs=max_pairs)
    fig, ax = plt.subplots(figsize=(2.12, 2.12))
    draw_segments(ax, x, y, pairs, min_width=0.07, max_width=1.05, alpha=0.58, zorder=1)
    draw_clouds(ax, x, y, size=POINT_SIZE, zorder=3)
    set_cloud_limits(ax, x, y, pad=0.18)
    save_pdf(fig, out / filename)
    plt.close(fig)